In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def bronze(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/bronze/{t}")

def to_silver(df, name):
    df.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/silver/{name}")

# tables with no special handling yet — just drop exact duplicate rows
passthrough = ["competitions", "clubs", "countries", "national_teams",
               "club_games", "game_events", "game_lineups", "transfers"]

for t in passthrough:
    to_silver(bronze(t).dropDuplicates(), t)

# tables that need explicit date casting + a real key-based dedupe
players = (bronze("players")
           .dropDuplicates(["player_id"])
           .withColumn("date_of_birth", F.to_date("date_of_birth")))
to_silver(players, "players")
print("players: date_of_birth nulls after cast:",
      players.filter(F.col("date_of_birth").isNull()).count())

games = (bronze("games")
         .dropDuplicates(["game_id"])
         .withColumn("date", F.to_date("date")))
to_silver(games, "games")
print("games: date nulls after cast:", games.filter(F.col("date").isNull()).count())

player_valuations = bronze("player_valuations").withColumn("date", F.to_date("date"))
to_silver(player_valuations, "player_valuations")

appearances = bronze("appearances").withColumn("date", F.to_date("date"))
to_silver(appearances, "appearances")